# DataMate – Inference trên Kaggle (bản sửa)

**Cài đặt Kaggle:** Accelerator = **GPU T4 x2**, Internet = **On**.

Những lỗi đã sửa so với bản cũ:
1. Nhánh `HoangMinh` mới nhất (pipeline v2) **không còn backend `llama_cpp`** và **không có `scripts/download_model.py`** → tải model bằng `huggingface_hub` và vá `src/datamate/llm.py` để thêm backend `llama_cpp`.
2. Cell chạy thử cũ dùng `configs/prod.yaml` (vLLM + `/models/Qwen2.5-14B`, không có trên Kaggle) → dùng `configs/kaggle.yaml` sinh từ `prod.yaml`.
3. Đường dẫn output/evaluate bị lệch (`train_10` vs `train_5`) → thống nhất.
4. Máy có **2 GPU T4** nhưng bản cũ chỉ chạy 1 tiến trình → chia public test theo business thành 2 shard, mỗi GPU 1 tiến trình, rồi gộp lại.
5. Tắt `UserWarning` lặp từ `db.py` làm tràn log.

**Bản vá CUDA/LLM:** Cell 3 cài CUDA 12 runtime + cuBLAS, đăng ký thư mục thư viện và kiểm tra `GPU offload`. Cell 5 preload CUDA shared libraries trước khi import `llama_cpp`. Cell 16 xóa `outputs/public` cũ trước khi chạy lại để không tái sử dụng submission chứa `abstain` từ lần chạy lỗi.


In [1]:
# Cell 1: Clone code (Kaggle: bật Internet). Chạy lại cell này sẽ xoá bản clone cũ
import os, shutil
os.chdir("/kaggle/working")
shutil.rmtree("zalo_challenge_2026_datamate", ignore_errors=True)
!git clone -q --single-branch --branch HoangMinh https://github.com/VanHung-05/zalo_challenge_2026_datamate.git
os.chdir("/kaggle/working/zalo_challenge_2026_datamate")
!git log --oneline -1

7381c3c (HEAD -> HoangMinh, origin/HoangMinh) feat(datamate): implement v2 pipeline


In [2]:
# Cell 2: Cài thư viện (dùng sys.executable để cài đúng Python chạy runner)
import sys
!{sys.executable} -m pip install -q -r requirements.txt
!{sys.executable} -m pip install -q diskcache jinja2 typing-extensions huggingface_hub  # deps của llama-cpp-python (cài --no-deps ở cell sau)

In [3]:
# Cell 3: Cài CUDA 12 runtime + llama-cpp-python CUDA wheel
# Lưu ý: chạy cell này sau Cell 2 và restart session nếu pip yêu cầu.

import os
import sys
import glob
from pathlib import Path

# 1. Cài CUDA runtime và cuBLAS cần thiết cho llama.cpp
!{sys.executable} -m pip install -q --upgrade nvidia-cuda-runtime-cu12 nvidia-cublas-cu12

# 2. Tìm thư viện trong các thư mục dist-packages/site-packages của Kaggle
cuda_lib_dirs = set()

patterns = [
    "/usr/local/lib/python*/dist-packages/nvidia/*/lib",
    "/usr/local/lib/python*/site-packages/nvidia/*/lib",
    "/usr/local/cuda*/lib64",
]

for pattern in patterns:
    for folder in glob.glob(pattern):
        if os.path.isdir(folder):
            cuda_lib_dirs.add(os.path.realpath(folder))

print("[CUDA] Library directories:")
for folder in sorted(cuda_lib_dirs):
    print(" -", folder)

# Đăng ký các thư mục để linker hệ thống tìm thấy libcudart.so.12
conf_path = "/etc/ld.so.conf.d/cuda-llama.conf"
with open(conf_path, "w", encoding="utf-8") as f:
    f.write("\n".join(sorted(cuda_lib_dirs)) + "\n")

!ldconfig

# 3. Cài llama-cpp-python CUDA 12.1 wheel
!{sys.executable} -m pip install -q llama-cpp-python --no-deps --force-reinstall --no-cache-dir --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121

# 4. Kiểm tra thư viện CUDA và khả năng GPU offload
import ctypes

runtime_candidates = []
for folder in cuda_lib_dirs:
    runtime_candidates.extend(glob.glob(os.path.join(folder, "libcudart.so.12*")))

if runtime_candidates:
    runtime_path = sorted(runtime_candidates)[0]
    ctypes.CDLL(runtime_path, mode=ctypes.RTLD_GLOBAL)
    print("[CUDA] Loaded runtime:", runtime_path)
else:
    print("[CUDA WARNING] Không tìm thấy libcudart.so.12 trong các thư mục đã quét.")

import llama_cpp
gpu_offload = llama_cpp.llama_supports_gpu_offload()
print(
    ">>> llama_cpp version:",
    llama_cpp.__version__,
    "| GPU offload:",
    gpu_offload
)

if not gpu_offload:
    raise RuntimeError(
        "llama-cpp-python đã import được nhưng không hỗ trợ GPU. "
        "Không chạy inference vì có thể âm thầm chuyển sang CPU."
    )

!nvidia-smi --query-gpu=name,memory.total --format=csv

[CUDA] Library directories:
 - /usr/local/cuda-13.0/targets/x86_64-linux/lib
 - /usr/local/lib/python3.13/dist-packages/nvidia/cu13/lib
 - /usr/local/lib/python3.13/dist-packages/nvidia/cublas/lib
 - /usr/local/lib/python3.13/dist-packages/nvidia/cuda_cupti/lib
 - /usr/local/lib/python3.13/dist-packages/nvidia/cuda_nvrtc/lib
 - /usr/local/lib/python3.13/dist-packages/nvidia/cuda_runtime/lib
 - /usr/local/lib/python3.13/dist-packages/nvidia/cudnn/lib
 - /usr/local/lib/python3.13/dist-packages/nvidia/cufft/lib
 - /usr/local/lib/python3.13/dist-packages/nvidia/curand/lib
 - /usr/local/lib/python3.13/dist-packages/nvidia/cusolver/lib
 - /usr/local/lib/python3.13/dist-packages/nvidia/cusparse/lib
 - /usr/local/lib/python3.13/dist-packages/nvidia/cusparselt/lib
 - /usr/local/lib/python3.13/dist-packages/nvidia/nccl/lib
 - /usr/local/lib/python3.13/dist-packages/nvidia/nvjitlink/lib
 - /usr/local/lib/python3.13/dist-packages/nvidia/nvshmem/lib
/sbin/ldconfig.real: /usr/local/lib/libtbbbind_2_

ggml_cuda_init: found 2 CUDA devices (Total VRAM: 29823 MiB):
  Device 0: Tesla T4, compute capability 7.5, VMM: yes, VRAM: 14911 MiB
  Device 1: Tesla T4, compute capability 7.5, VMM: yes, VRAM: 14911 MiB


>>> llama_cpp version: 0.3.36 | GPU offload: True
name, memory.total [MiB]
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB


In [4]:
# Cell 4 sửa lại:
from huggingface_hub import hf_hub_download
REPO = "Qwen/Qwen2.5-Coder-7B-Instruct-GGUF"
FILE = "qwen2.5-coder-7b-instruct-q4_k_m.gguf"
MODEL_PATH = hf_hub_download(repo_id=REPO, filename=FILE, local_dir="/kaggle/working/model")
print(MODEL_PATH, round(os.path.getsize(MODEL_PATH) / 1e6), "MB")

qwen2.5-coder-7b-instruct-q4_k_m.gguf: reconstructing file:   0%|          |  0.00B / 4.68GB            

qwen2.5-coder-7b-instruct-q4_k_m.gguf: downloading bytes:           |  0.00B            

/kaggle/working/model/qwen2.5-coder-7b-instruct-q4_k_m.gguf 4683 MB


In [5]:
%%writefile src/datamate/llm.py
"""LLM Wrapper supporting dual backends:
- Ollama: development on Mac
- llama.cpp: Kaggle / single GPU GGUF
- vLLM: production on GPU
"""

from typing import Any, Dict, List, Optional
import requests


class LLMWrapper:
    """Unified LLM inference interface.

    Dev:
        Ollama + qwen2.5:7b / qwen3:8b over local REST API.

    Kaggle:
        llama.cpp + GGUF on a single GPU.

    Production:
        vLLM + Qwen2.5-14B / Qwen3-14B with Tensor Parallelism.
    """

    def __init__(self, config: Dict[str, Any]):
        llm_cfg = config.get("llm", {})

        self.backend = llm_cfg.get("backend", "ollama")
        self.model = llm_cfg.get("model", "qwen2.5:7b")
        self.seed = int(llm_cfg.get("seed", 42))
        self.default_temp = float(llm_cfg.get("temperature", 0.0))
        self.default_max_tokens = int(llm_cfg.get("max_tokens", 4096))

        self.engine = None

        # =====================================================
        # 1. OLLAMA BACKEND
        # =====================================================
        if self.backend == "ollama":
            self.base_url = llm_cfg.get("ollama_url", "http://localhost:11434")
            self.num_ctx = int(llm_cfg.get("num_ctx", 32768))
            self.request_timeout = int(llm_cfg.get("request_timeout", 300))

        # =====================================================
        # 2. LLAMA.CPP BACKEND
        # =====================================================
        elif self.backend == "llama_cpp":
            try:
                import ctypes
                import glob
                import os

                cuda_patterns = [
                    "/usr/local/lib/python*/dist-packages/nvidia/*/lib/libcudart.so.12*",
                    "/usr/local/lib/python*/site-packages/nvidia/*/lib/libcudart.so.12*",
                    "/usr/local/lib/python*/dist-packages/nvidia/*/lib/libcublasLt.so.12*",
                    "/usr/local/lib/python*/site-packages/nvidia/*/lib/libcublasLt.so.12*",
                    "/usr/local/lib/python*/dist-packages/nvidia/*/lib/libcublas.so.12*",
                    "/usr/local/lib/python*/site-packages/nvidia/*/lib/libcublas.so.12*",
                    "/usr/local/cuda*/lib64/libcudart.so.12*",
                    "/usr/local/cuda*/lib64/libcublasLt.so.12*",
                    "/usr/local/cuda*/lib64/libcublas.so.12*",
                ]

                preloaded = set()
                for pattern in cuda_patterns:
                    for lib_path in sorted(glob.glob(pattern)):
                        real_path = os.path.realpath(lib_path)
                        if real_path in preloaded:
                            continue
                        try:
                            ctypes.CDLL(real_path, mode=ctypes.RTLD_GLOBAL)
                            preloaded.add(real_path)
                        except OSError as load_error:
                            print(
                                f"[LLM Warning] Could not preload {real_path}: {load_error}",
                                flush=True
                            )

                from llama_cpp import Llama
            except ImportError:
                raise ImportError(
                    "llama-cpp-python is not installed. "
                    "On Kaggle, install a CUDA-enabled wheel first."
                )

            import os

            model_path = llm_cfg.get("model_path")
            if not model_path or not os.path.exists(model_path):
                raise FileNotFoundError(
                    f"GGUF model not found: {model_path}. "
                    "Run the model download cell first."
                )

            self.enable_thinking = bool(llm_cfg.get("enable_thinking", False))

            kw = {
                "model_path": model_path,
                "n_ctx": int(llm_cfg.get("n_ctx", 32768)),
                "n_gpu_layers": int(llm_cfg.get("n_gpu_layers", -1)),
                "n_batch": int(llm_cfg.get("n_batch", 1024)),
                "seed": self.seed,
                "verbose": bool(llm_cfg.get("verbose", False)),
            }
            if "flash_attn" in llm_cfg:
                kw["flash_attn"] = bool(llm_cfg["flash_attn"])

            self.engine = Llama(**kw)

        # =====================================================
        # 3. vLLM BACKEND
        # =====================================================
        elif self.backend == "vllm":
            try:
                from vllm import LLM

                model_path = llm_cfg.get("model_path", self.model)
                tp_size = int(llm_cfg.get("tp_size", 2))

                self.engine = LLM(
                    model=model_path,
                    tensor_parallel_size=tp_size,
                    seed=self.seed,
                    trust_remote_code=True,
                )
            except ImportError:
                raise ImportError(
                    "vLLM is not installed. "
                    "Use 'ollama' for Mac dev or 'llama_cpp' for Kaggle."
                )

        else:
            raise ValueError(f"Unknown LLM backend: {self.backend}")

    # =========================================================
    # CHAT TEMPLATE HELPER
    # =========================================================
    def _chat(self, prompt: str) -> str:
        """Wrap prompt in the target model's chat template."""
        if self.backend == "llama_cpp":
            return f"<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n"

        if self.backend == "vllm":
            try:
                tok = self.engine.get_tokenizer()
                if getattr(tok, "chat_template", None):
                    return tok.apply_chat_template(
                        [{"role": "user", "content": prompt}],
                        tokenize=False,
                        add_generation_prompt=True,
                    )
            except Exception as e:
                print(f"[LLM Warning] vLLM chat template not applied: {e}")

        return prompt

    # =========================================================
    # GENERATE
    # =========================================================
    def generate(
        self,
        prompt: str,
        temperature: Optional[float] = None,
        max_tokens: Optional[int] = None,
        n: int = 1,
        stop: Optional[List[str]] = None,
        seed: Optional[int] = None,
    ) -> List[str]:
        """Generate one or multiple responses.
        Each sample uses base_seed + i for reproducibility.
        """
        if n < 1:
            raise ValueError("n must be >= 1")

        temp = temperature if temperature is not None else self.default_temp
        max_tok = max_tokens if max_tokens is not None else self.default_max_tokens
        stop_tokens = (
            stop
            if stop is not None
            else ["```\n\n", "\n```\n", "<|im_end|>", "<|im_start|>", "\n\n\n\n"]
        )

        req_seed = self.seed if seed is None else int(seed)

        # 1. OLLAMA
        if self.backend == "ollama":
            responses = []
            for i in range(n):
                payload = {
                    "model": self.model,
                    "prompt": self._chat(prompt),
                    "options": {
                        "temperature": temp,
                        "num_predict": max_tok,
                        "num_ctx": self.num_ctx,
                        "seed": req_seed + i,
                        "stop": stop_tokens,
                    },
                    "stream": False,
                }
                try:
                    resp = requests.post(
                        f"{self.base_url}/api/generate",
                        json=payload,
                        timeout=self.request_timeout,
                    )
                    resp.raise_for_status()
                    data = resp.json()
                    responses.append(data.get("response", ""))
                except Exception as e:
                    print(f"[LLM Error] Ollama request failed: {e}", flush=True)
                    responses.append("")
            return responses

        # 2. LLAMA.CPP
        elif self.backend == "llama_cpp":
            text_prompt = self._chat(prompt)
            responses = []
            for i in range(n):
                try:
                    output = self.engine.create_completion(
                        prompt=text_prompt,
                        max_tokens=max_tok,
                        temperature=temp,
                        top_p=0.95 if temp > 0 else 1.0,
                        stop=stop_tokens,
                        seed=req_seed + i,
                    )
                    responses.append(output["choices"][0]["text"])
                except Exception as e:
                    print(f"[LLM Error] llama.cpp generation failed: {e}", flush=True)
                    responses.append("")
            return responses

        # 3. vLLM
        elif self.backend == "vllm":
            from vllm import SamplingParams

            params = SamplingParams(
                temperature=temp,
                max_tokens=max_tok,
                n=n,
                stop=stop_tokens,
                seed=req_seed,
            )
            text_prompt = self._chat(prompt)
            try:
                outputs = self.engine.generate([text_prompt], params)
                return [item.text for item in outputs[0].outputs]
            except Exception as e:
                print(f"[LLM Error] vLLM generation failed: {e}", flush=True)
                return [""] * n

        return []

Overwriting src/datamate/llm.py


In [6]:
# Cell 6 sửa lại:
import yaml
cfg = yaml.safe_load(open("configs/prod.yaml", encoding="utf-8"))

cfg["llm"] = {
    "backend": "llama_cpp",
    "model": "Qwen/Qwen2.5-Coder-7B-Instruct-GGUF",
    "model_path": MODEL_PATH,
    "n_ctx": 32768,
    "n_gpu_layers": -1,
    "n_batch": 1024,          # Tăng lên 1024 để GPU nạp prompt nhanh gấp đôi
    "flash_attn": True,
    "enable_thinking": False,
    "seed": 42,
    "temperature": 0.0,
    "max_tokens": 1024,
}
cfg["agent"].update({
    "n_samples": 1, 
    "max_retries": 3,
    "sandbox_timeout": 60,
})

# Đặt tau = 0.52 (với p_v heuristic = 0.60, vừa đủ pass 1 LLM sample nhưng giảm bớt false positive)
cfg["decision"]["tau"] = 0.52

cfg["runner"] = {
    "time_budget_seconds": 36000,
    "question_timeout_seconds": 180,
    "cache_dir": "outputs/cache",
    "parallel_workers": 1,
    "group_by_business": True,
    "data_dir": "train",
    "output_file": "submission.jsonl",
}
yaml.safe_dump(cfg, open("configs/kaggle.yaml", "w", encoding="utf-8"), allow_unicode=True, sort_keys=False)

## Chạy thử 10 câu train (1 GPU)
Đầu output phải có `[LLM] llama.cpp device: GPU`. Xem thời gian/câu trước khi chạy cả bộ.

In [7]:
env = "PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0"
# # !{env} {sys.executable} src/datamate/runner.py -c configs/kaggle.yaml -d train --limit 10 -o outputs/train_10/submission.jsonl 2>&1 | grep -v "UserWarning\|MILLION_SHORTHAND_RE"

# !python src/datamate/runner.py -c configs/kaggle.yaml -d train --limit 300 -o outputs/train_300/submission.jsonl!
# !python scripts/train_calibrator.py --logs outputs/train_300/logs -g train/answers.jsonl -q train/questions.jsonl --save configs/calibrator.json


In [8]:
# !{sys.executable} scripts/evaluate.py -p outputs/train_300/submission.jsonl -g train/answers.jsonl --only-predicted

In [9]:
# # Xem log 1 câu: kết quả cuối + thời gian + trace
# import glob, json
# f = sorted(glob.glob("outputs/train_10/logs/q_*.json"))[0]
# log = json.load(open(f, encoding="utf-8"))
# print(log["question"], "->", log["final"], f"({log['elapsed_seconds']}s)")
# print(json.dumps({k: v for k, v in log.items() if k not in ("question", "final")}, ensure_ascii=False, indent=1, default=str)[:3000])

## Inference public test (1000 câu) – chia 2 GPU
Ghi từng dòng + `--resume`: nếu bị ngắt giữa chừng, chạy lại 2 cell kế tiếp là tiếp tục từ chỗ cũ.

In [10]:
# # Chia public_test thành 2 shard theo business_id (các câu cùng business ở cùng 1 GPU), cân bằng số câu
# import json, collections
# SRC = "/kaggle/working/zalo_challenge_2026_datamate/public_test"
# SH = "/kaggle/working/shards"
# N_SHARDS = 2
# qs = [json.loads(l) for l in open(f"{SRC}/questions.jsonl", encoding="utf-8") if l.strip()]
# by_biz = collections.defaultdict(list)
# for q in qs: by_biz[q["business_id"]].append(q)
# shards = [[] for _ in range(N_SHARDS)]
# for biz, items in sorted(by_biz.items(), key=lambda kv: -len(kv[1])):
#     min(shards, key=len).extend(items)
# shutil.rmtree(SH, ignore_errors=True)
# for i, items in enumerate(shards):
#     d = f"{SH}/s{i}"; os.makedirs(d)
#     for name in os.listdir(SRC):                      # symlink data/, calendar.csv, rulebook.md ...
#         if name != "questions.jsonl": os.symlink(f"{SRC}/{name}", f"{d}/{name}")
#     with open(f"{d}/questions.jsonl", "w", encoding="utf-8") as f:
#         for q in items: f.write(json.dumps(q, ensure_ascii=False) + "\n")
#     print(f"shard {i}: {len(items)} câu, {len({q['business_id'] for q in items})} business")

In [11]:
# # Chạy 2 tiến trình nền, mỗi tiến trình 1 GPU
# import subprocess, time
# ROOT = "/kaggle/working/zalo_challenge_2026_datamate"
# procs = []
# for i in range(N_SHARDS):
#     out = f"{ROOT}/outputs/public/s{i}/submission.jsonl"
#     os.makedirs(os.path.dirname(out), exist_ok=True)
#     env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(i), "PYTHONWARNINGS": "ignore", "PYTHONUNBUFFERED": "1"}
#     logf = open(f"{ROOT}/outputs/public/run_s{i}.log", "a")
#     procs.append(subprocess.Popen(
#         [sys.executable, "src/datamate/runner.py", "-c", "configs/kaggle.yaml", "-d", f"{SH}/s{i}", "-o", out, "--resume"],
#         cwd=ROOT, env=env, stdout=logf, stderr=subprocess.STDOUT))
# print("đã chạy:", [p.pid for p in procs])

In [12]:
# # Theo dõi tiến độ; cell kết thúc khi cả 2 tiến trình xong (có thể Interrupt cell này, tiến trình nền vẫn chạy)
# def progress():
#     res = []
#     for i in range(N_SHARDS):
#         p = f"{ROOT}/outputs/public/s{i}/submission.jsonl"
#         res.append(sum(1 for _ in open(p, encoding="utf-8")) if os.path.exists(p) else 0)
#     return res
# t0 = time.time()
# while any(p.poll() is None for p in procs):
#     time.sleep(120)
#     done = progress()
#     print(f"[{(time.time()-t0)/60:6.1f} phút] xong {sum(done)}/{len(qs)}  (theo shard: {done})", flush=True)
# print("returncode:", [p.returncode for p in procs], "| xong:", progress())

In [13]:
# # Gộp shard -> submission.jsonl; câu nào thiếu thì điền abstain để file luôn đủ 1000 dòng
# final = {}
# for i in range(N_SHARDS):
#     p = f"{ROOT}/outputs/public/s{i}/submission.jsonl"
#     if os.path.exists(p):
#         for l in open(p, encoding="utf-8"):
#             try: r = json.loads(l)
#             except json.JSONDecodeError: continue
#             final[r["id"]] = r
# missing = [q["id"] for q in qs if q["id"] not in final]
# for qid in missing: final[qid] = {"id": qid, "type": "abstain"}
# os.makedirs(f"{ROOT}/outputs/public", exist_ok=True)
# with open(f"{ROOT}/outputs/public/submission.jsonl", "w", encoding="utf-8") as f:
#     for q in qs: f.write(json.dumps(final[q["id"]], ensure_ascii=False) + "\n")
# print(f"tổng {len(final)} dòng, thiếu (điền abstain): {len(missing)}")
# !{sys.executable} scripts/validate_submission.py -p outputs/public/submission.jsonl -q public_test/questions.jsonl

In [14]:
# Xóa submission cũ trước khi chạy lại để --resume không giữ các dòng abstain từ lần chạy lỗi.
import shutil
shutil.rmtree("outputs/public", ignore_errors=True)

# Chạy inference mới
!{sys.executable} src/datamate/runner.py -c configs/kaggle.yaml -d public_test -o outputs/public/submission.jsonl --resume

# Kiểm tra định dạng và số lượng kết quả
!{sys.executable} scripts/validate_submission.py -p outputs/public/submission.jsonl -q public_test/questions.jsonl


[LLM] llama.cpp device: GPU
Loaded 1000 questions from public_test/questions.jsonl (mode=sql, pipeline=v2)
Output will be saved to: outputs/public/submission.jsonl
Per-question logs: /kaggle/working/zalo_challenge_2026_datamate/outputs/public/logs
[Load] dev_001 in 6.8s
[1/1000] [dev_001] (7.1s) "Doanh thu của Bia Heineken 330ml trong tháng 7 năm 2025 l..."
       -> [number]: 3142000
[2/1000] [dev_001] (15.8s) "So sánh 3/25, 4/25 và 5/25 xem tháng nào cửa hàng bán đc ..."
       -> [abstain]: abstain
[3/1000] [dev_001] (0.0s) "Ai là khách có số đơn mua nhiều nhất trong tháng 4 năm 2025?"
       -> [string]: La Minh Dương
[4/1000] [dev_001] (2.6s) "Từ đầu năm tới hết hnay tiệm có tổng cộng bnhiu đh rồi?"
       -> [abstain]: abstain
[5/1000] [dev_001] (3.9s) "T5/2025 trung bình mỗi đơn giá trị bnhiu?"
       -> [abstain]: abstain
[6/1000] [dev_001] (3.1s) "Khách mua ít nhất 4 đơng trong tháng 11/2025 có mấy người?"
       -> [abstain]: abstain
[7/1000] [dev_001] (0.0s) "Tu dau nam toi 

In [15]:
# Copy file nộp ra /kaggle/working để tải từ tab Output
!cp outputs/public/submission.jsonl /kaggle/working/submission.jsonl && ls -la /kaggle/working/submission.jsonl

-rw-r--r-- 1 root root 54809 Nov  3 19:27 /kaggle/working/submission.jsonl


In [ ]:
# Cell 18: Thống kê và kiểm tra kết quả
import json, collections

with open("outputs/public/submission.jsonl", encoding="utf-8") as f:
    rows = [json.loads(line) for line in f]

print(f"Tổng số câu: {len(rows)}")
types = collections.Counter(r.get("type") for r in rows)
print("\nPhân phối đáp án:")
for t, cnt in types.most_common():
    print(f"  {t:15s}: {cnt:4d} ({cnt/len(rows)*100:.1f}%)")

abstains = [r for r in rows if r.get("type") == "abstain"]
print(f"\nTổng abstain: {len(abstains)} ({len(abstains)/len(rows)*100:.1f}%)")
print(f"Tổng có câu trả lời: {len(rows) - len(abstains)} ({(len(rows) - len(abstains))/len(rows)*100:.1f}%)")

# Kiểm tra các câu không abstain xem có giá trị hợp lệ không
null_answers = [r for r in rows if r.get("type") != "abstain" and r.get("answer") is None and r.get("type") != "unanswerable"]
print(f"Số câu lỗi giá trị None: {len(null_answers)}")